# Programming Assignment (HDS - 5230 Spring 2025) - Week 12 - Neural Network

### Author : Daipayan Bera
### Date : 04-05-2025

## This exercise will entail the creation of neural networks with 2 configurations and the efficacy of that on different sizes of datasets, which will be generated through a data synthesizer.

In [16]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.utils import resample

df = pd.read_csv("PimaIndiansDiabetes2.csv") #I had exported the pimaIndian dataset through R and imported directly here.
ds = df.dropna()

# 2. Fit logistic regression model
# Note: We'll use sklearn which automatically handles logistic regression
X = ds.drop(columns=['diabetes'])
y = (ds['diabetes'] == 'pos').astype(int)  # convert to 0/1

logmodel = LogisticRegression(max_iter=500)
logmodel.fit(X, y)

cfs = np.concatenate(([logmodel.intercept_[0]], logmodel.coef_[0]))  # first intercept, then coefficients
prednames = X.columns.tolist()

def data_generator(sz):
    """ This function will enable us to create synthesized dataset from PimaIndiansDiabetes2.csv dataset, in which 
    we have performed few operations. With that, we are able to generate any number of data from the given dataset with 
    this function"""
    dfdata = pd.DataFrame({
        name: resample(ds[name], n_samples=sz, replace=True, random_state=None).reset_index(drop=True)
        for name in prednames
    })

    # Compute the logit (linear combination of predictors and coefficients)
    pvec = sum(
        cfs[i+1] * dfdata[prednames[i]] for i in range(len(prednames))
    ) + cfs[0]

    # Compute probability using sigmoid function
    probs = 1 / (1 + np.exp(-pvec))

    # Generate outcomes based on probability
    dfdata['outcome'] = (probs > 0.5).astype(int)
    
    return dfdata

In [17]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

def processdata(size):
    data = data_generator(size)
    X = data.iloc[:, 1:8]  
    y = data["outcome"].values
    X_train, X_test, y_train, y_test = train_test_split(X,y, random_state=32, test_size=0.2)
    trans_1 = Pipeline([("impute", SimpleImputer(strategy="mean")),
                      ("scale", StandardScaler())]).fit(X_train)
    X_train_s = trans_1.transform(X_train)
    test_1 = Pipeline([("impute", SimpleImputer(strategy="mean")),
                      ("scale", StandardScaler())]).fit(X_test)
    X_test_s = test_1.transform(X_test)
    return X_train_s, X_test_s, y_train, y_test

In [18]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
import tensorflow as tf
metrics = [tf.keras.metrics.BinaryAccuracy(name='accuracy'),
           tf.keras.metrics.AUC(name='auc')]
def build_model1(t_shape):
    """ This is a neural network model with 1 hidden layer 4 nodes configuration for classification operation"""
    model = Sequential()
    model.add(Input(shape=(t_shape,)))
    model.add(Dense(4,activation='relu'))
    model.add(Dense(1, activation='sigmoid'))
    model.compile(optimizer='rmsprop',
                  loss='binary_crossentropy',
                  metrics=metrics)  
    return model

def build_model2(t_shape):
    """ This is another neural network model with 2 hidden layers of 4 nodes each configuration for classification operation"""
    model = Sequential()
    model.add(Input(shape=(t_shape,)))
    model.add(Dense(4, activation='relu'))
    model.add(Dense(4, activation='relu'))
    model.add(Dense(1, activation='sigmoid'))
    model.compile(optimizer='rmsprop',
                  loss='binary_crossentropy', 
                  metrics=metrics) 
    return model

In [19]:
#Creating result dataframe to be able display the result
result = pd.DataFrame(columns=["Data size", "Configuration", "Training Error", "Validation Error", "Time of execution", "Test Error"])

In [20]:
import timeit
from sklearn.metrics import accuracy_score
datasize = [1000,10000,100000]
pos = 0
for sz in datasize:
    X_train, X_test, y_train, y_test = processdata(sz)
    class1 = build_model1(X_train.shape[1])
    execution_time = timeit.timeit(lambda: class1.fit(X_train, y_train,
                                                   epochs=100,
                                                   validation_split=0.2,
                                                   verbose=0),
                                                   number=1)
    y_pred_prob = class1.predict(X_test)
    y_pred = (y_pred_prob > 0.5).astype(int)
    result.loc[pos] = [sz, "1 hidden layer 4 nodes", round(np.max(class1.history.history["accuracy"]),2), round(np.max(class1.history.history["val_accuracy"]),2), execution_time, round(accuracy_score(y_test, y_pred),2)]
    pos+=1

7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step  
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
625/625 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step


In [21]:
for sz in datasize:
    class2 = build_model2(X_train.shape[1])
    X_train, X_test, y_train, y_test = processdata(sz)
    execution_time = timeit.timeit(lambda: class2.fit(X_train, y_train,
                                                   epochs=100,
                                                   validation_split=0.2,
                                                   verbose=0),
                                                   number=1)
    y_pred_prob = class2.predict(X_test)
    y_pred = (y_pred_prob > 0.5).astype(int)
    result.loc[pos] = [sz, "2 hidden layers of 4 nodes each", round(np.max(class2.history.history["accuracy"]),2), round(np.max(class2.history.history["val_accuracy"]),2), execution_time, round(accuracy_score(y_test, y_pred),2)]
    pos+=1

7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step  
625/625 ━━━━━━━━━━━━━━━━━━━━ 1s 899us/step


In [23]:
result

,Data size,Configuration,Training Error,Validation Error,Time of execution,Test Error
0,1000,1 hidden layer 4 nodes,0.96,0.96,15.823973,0.94
1,10000,1 hidden layer 4 nodes,0.96,0.97,49.841671,0.96
2,100000,1 hidden layer 4 nodes,0.96,0.96,466.922261,0.96
3,1000,2 hidden layers of 4 nodes each,0.95,0.93,16.872266,0.92
4,10000,2 hidden layers of 4 nodes each,0.97,0.96,54.947349,0.96
5,100000,2 hidden layers of 4 nodes each,0.96,0.96,450.662969,0.96
